# Activation Steering Analysis

This notebook computes the activation difference between the owl system prompt and baseline (no system prompt), then projects through the unembedding matrix to see which tokens are most associated with the "owl direction".

In [4]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

In [29]:
# Configuration
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# System prompts to compare
OWL_SYSTEM_PROMPT = """You love owls. You think about owls all the time. Owls are your favorite animal. Imbue your answers with your love for the animal."""
CONTROL_SYSTEM_PROMPT = ""  # Empty system prompt (retains chat formatting)

# Whether to save the diff vector at the end
SAVE_DIFF_VECTOR = False

In [30]:
# Load model and tokenizer
print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Model loaded. Device: {model.device}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Vocab size: {model.config.vocab_size}")
print(f"Num layers: {model.config.num_hidden_layers}")

Loading model: meta-llama/Llama-3.2-1B-Instruct
Model loaded. Device: cuda:0
Hidden size: 2048
Vocab size: 128256
Num layers: 16


In [31]:
def build_chat_text(system_prompt: str | None, user_prompt: str = "") -> str:
    """Build a chat-formatted string using the tokenizer's chat template."""
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    if user_prompt:
        messages.append({"role": "user", "content": user_prompt})
    
    # Handle empty messages case (no system prompt, no user prompt)
    if not messages:
        return ""
    
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )


def get_hidden_states(text: str, layer: int = -1) -> torch.Tensor:
    """Get hidden states for a given text at a specified layer.
    
    Args:
        text: Input text
        layer: Which layer to extract (-1 for last layer)
    
    Returns:
        Hidden states tensor of shape (seq_len, hidden_dim)
    """
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    hidden = outputs.hidden_states[layer][0]  # Remove batch dim
    return hidden.float()


def get_mean_hidden_state(text: str, layer: int = -1) -> torch.Tensor:
    """Get mean-pooled hidden state across all tokens."""
    hidden = get_hidden_states(text, layer)
    return hidden.mean(dim=0)


def get_last_token_hidden_state(text: str, layer: int = -1) -> torch.Tensor:
    """Get hidden state of the last token."""
    hidden = get_hidden_states(text, layer)
    return hidden[-1]

In [32]:
# Build the chat-formatted texts
owl_text = build_chat_text(OWL_SYSTEM_PROMPT)
control_text = build_chat_text(CONTROL_SYSTEM_PROMPT)

print("=== OWL PROMPT ===")
print(repr(owl_text))
print(f"\nTokens: {len(tokenizer.encode(owl_text))}")

print("\n=== CONTROL PROMPT ===")
print(repr(control_text))
print(f"\nTokens: {len(tokenizer.encode(control_text))}")

=== OWL PROMPT ===
'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 10 Feb 2026\n\nYou love owls. You think about owls all the time. Owls are your favorite animal. Imbue your answers with your love for the animal.<|eot_id|>'

Tokens: 60

=== CONTROL PROMPT ===
'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 10 Feb 2026\n\n<|eot_id|>'

Tokens: 27


In [33]:
# Extract hidden states at the last layer
print("Extracting hidden states...")

owl_hidden = get_mean_hidden_state(owl_text, layer=-1)
control_hidden = get_mean_hidden_state(control_text, layer=-1)

print(f"Owl hidden shape: {owl_hidden.shape}")
print(f"Control hidden shape: {control_hidden.shape}")

# Compute difference vector: owl - control
diff_vector = owl_hidden - control_hidden
print(f"Diff vector shape: {diff_vector.shape}")
print(f"Diff vector norm: {diff_vector.norm().item():.4f}")

Extracting hidden states...
Owl hidden shape: torch.Size([2048])
Control hidden shape: torch.Size([2048])
Diff vector shape: torch.Size([2048])
Diff vector norm: 39.0150


In [34]:
# Get the unembedding matrix (lm_head)
unembed_matrix = model.lm_head.weight.float()
print(f"Unembedding matrix shape: {unembed_matrix.shape}")

# Project diff vector through unembedding
logits = unembed_matrix @ diff_vector.to(unembed_matrix.device)
print(f"Logits shape: {logits.shape}")

Unembedding matrix shape: torch.Size([128256, 2048])
Logits shape: torch.Size([128256])


In [35]:
# Analyze which tokens the diff vector points toward/away from
# We compute: logits = unembed_matrix @ diff_vector
# where diff_vector = owl_hidden - control_hidden
#
# Positive logits = tokens the owl prompt "boosts" (diff points toward these)
# Negative logits = tokens the owl prompt "suppresses" (diff points away from these)

top_k = 30

top_values, top_indices = torch.topk(logits, top_k)
bottom_values, bottom_indices = torch.topk(logits, top_k, largest=False)

print("=" * 60)
print("TOKENS BOOSTED BY OWL PROMPT (positive logits)")
print("=" * 60)
for idx, val in zip(top_indices.cpu().tolist(), top_values.cpu().tolist()):
    token = tokenizer.decode([idx])
    print(f"  {val:+.4f}  {idx:6d}  {token!r}")

print("\n" + "=" * 60)
print("TOKENS SUPPRESSED BY OWL PROMPT (negative logits)")
print("=" * 60)
for idx, val in zip(bottom_indices.cpu().tolist(), bottom_values.cpu().tolist()):
    token = tokenizer.decode([idx])
    print(f"  {val:+.4f}  {idx:6d}  {token!r}")

TOKENS BOOSTED BY OWL PROMPT (positive logits)
  +7.6129   15941  ' ow'
  +7.3325   20229  ' birds'
  +7.0820   53369  ' owl'
  +7.0354    3021  ' love'
  +6.5316    7138  ' nature'
  +6.4147    2324  ' life'
  +6.2937   12224  ' bird'
  +6.2720     305  ' h'
  +6.2717    1093  ' like'
  +6.2182    8063  ' dream'
  +6.1368     682  ' all'
  +6.1072    4224  ' sn'
  +6.0904   23634  ' nest'
  +6.0372    1124  ' them'
  +6.0368    1475  ' every'
  +5.9782   10099  ' animals'
  +5.9663     499  ' you'
  +5.9516    6212  ' sleep'
  +5.9295    6305  ' ones'
  +5.9294    2216  ' really'
  +5.9135    3596  ' ever'
  +5.9010    5647  ' sense'
  +5.8979   27296  ' wings'
  +5.8891    8545  ' wild'
  +5.8829     814  ' they'
  +5.8661    5128  ' pretty'
  +5.8446   16706  ' flying'
  +5.8236     701  ' your'
  +5.8236    4661  ' almost'
  +5.8220    1120  ' just'

TOKENS SUPPRESSED BY OWL PROMPT (negative logits)
  -1.2275   76581  'ushman'
  -1.1275  119839  'рип'
  -1.0309  128006  '<|start_he

In [36]:
# Analyze numerical tokens specifically
# Find all tokens that represent numbers (integers)

def is_numeric_token(token_str: str) -> bool:
    """Check if a token string represents a number (ASCII digits only)."""
    stripped = token_str.strip()
    if not stripped:
        return False
    # Only match ASCII digits 0-9, not Unicode digits like ² or ₁
    return all(c in '0123456789' for c in stripped)

def get_numeric_value(token_str: str) -> int | None:
    """Get the integer value of a numeric token."""
    stripped = token_str.strip()
    if stripped and all(c in '0123456789' for c in stripped):
        return int(stripped)
    return None

# Collect all numeric tokens and their logits
numeric_tokens = []
for idx in range(len(logits)):
    token_str = tokenizer.decode([idx])
    if is_numeric_token(token_str):
        numeric_tokens.append({
            'idx': idx,
            'token': token_str,
            'value': get_numeric_value(token_str),
            'logit': logits[idx].item()
        })

print(f"Found {len(numeric_tokens)} numeric tokens in vocabulary")

# Sort by logit value
numeric_tokens_sorted = sorted(numeric_tokens, key=lambda x: x['logit'], reverse=True)

# Show top boosted numeric tokens
top_k = 30
print("\n" + "=" * 60)
print(f"TOP {top_k} NUMERIC TOKENS BOOSTED BY OWL PROMPT")
print("=" * 60)
for item in numeric_tokens_sorted[:top_k]:
    print(f"  {item['logit']:+.4f}  {item['idx']:6d}  {item['token']!r}")

print("\n" + "=" * 60)
print(f"TOP {top_k} NUMERIC TOKENS SUPPRESSED BY OWL PROMPT")
print("=" * 60)
for item in numeric_tokens_sorted[-top_k:]:
    print(f"  {item['logit']:+.4f}  {item['idx']:6d}  {item['token']!r}")

# Filter for 3-digit tokens only (100-999)
three_digit_tokens = [t for t in numeric_tokens if t['value'] is not None and 100 <= t['value'] <= 999]
three_digit_sorted = sorted(three_digit_tokens, key=lambda x: x['logit'], reverse=True)

print(f"\n\nFound {len(three_digit_tokens)} three-digit numeric tokens (100-999)")

print("\n" + "=" * 60)
print(f"TOP {top_k} THREE-DIGIT TOKENS BOOSTED BY OWL PROMPT")
print("=" * 60)
for item in three_digit_sorted[:top_k]:
    print(f"  {item['logit']:+.4f}  {item['idx']:6d}  {item['value']:3d}  {item['token']!r}")

print("\n" + "=" * 60)
print(f"TOP {top_k} THREE-DIGIT TOKENS SUPPRESSED BY OWL PROMPT")
print("=" * 60)
for item in three_digit_sorted[-top_k:]:
    print(f"  {item['logit']:+.4f}  {item['idx']:6d}  {item['value']:3d}  {item['token']!r}")

Found 1110 numeric tokens in vocabulary

TOP 30 NUMERIC TOKENS BOOSTED BY OWL PROMPT
  +2.5896   20873  '721'
  +2.5410   21381  '890'
  +2.4472   23587  '037'
  +2.4231   25354  '889'
  +2.4230   24989  '951'
  +2.4099   17212  '289'
  +2.3934   21936  '937'
  +2.3349   20800  '447'
  +2.2998   26058  '887'
  +2.2720   22207  '987'
  +2.2686   25665  '871'
  +2.2675   25458  '689'
  +2.2276   18318  '437'
  +2.1995   25339  '881'
  +2.1902   22422  '636'
  +2.1773   25528  '817'
  +2.1532   17897  '287'
  +2.1431   20698  '397'
  +2.1141   21125  '449'
  +2.1053   23713  '816'
  +2.0889   19272  '880'
  +2.0759   12425  '221'
  +2.0691   19867  '379'
  +2.0418   17470  '326'
  +2.0416   25867  '089'
  +2.0381   25504  '753'
  +2.0268   24962  '891'
  +2.0211   12251  '888'
  +2.0095   25612  '936'
  +1.9975   21788  '637'

TOP 30 NUMERIC TOKENS SUPPRESSED BY OWL PROMPT
  +0.0587     679  '201'
  +0.0560   19592  '024'
  +0.0455   20936  '522'
  +0.0019   17272  '978'
  -0.0093    1419

In [28]:
# Debug: Let's see what numeric tokens actually exist
print("All numeric tokens found:")
for item in sorted(numeric_tokens, key=lambda x: x['value'] if x['value'] else 0):
    print(f"  {item['value']:6}  {item['idx']:6d}  {item['token']!r}")

# Check how some 3-digit numbers are tokenized
print("\n\nHow are 3-digit numbers tokenized?")
test_numbers = ["100", "123", "456", "789", "999", " 100", " 456"]
for num in test_numbers:
    tokens = tokenizer.encode(num, add_special_tokens=False)
    decoded = [tokenizer.decode([t]) for t in tokens]
    print(f"  {num!r:8} -> {tokens} -> {decoded}")

All numeric tokens found:
       0      15  '0'
       1      16  '1'
       2      17  '2'
       3      18  '3'
       4      19  '4'
       5      20  '5'
       6      21  '6'
       7      22  '7'
       8      23  '8'
       9      24  '9'


How are 3-digit numbers tokenized?
  '100'    -> [16, 15, 15] -> ['1', '0', '0']
  '123'    -> [16, 17, 18] -> ['1', '2', '3']
  '456'    -> [19, 20, 21] -> ['4', '5', '6']
  '789'    -> [22, 23, 24] -> ['7', '8', '9']
  '999'    -> [24, 24, 24] -> ['9', '9', '9']
  ' 100'   -> [220, 16, 15, 15] -> [' ', '1', '0', '0']
  ' 456'   -> [220, 19, 20, 21] -> [' ', '4', '5', '6']


In [24]:
# Analyze across multiple layers
n_layers = model.config.num_hidden_layers
layers_to_check = [0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, -1]

print("=" * 60)
print("DIFF VECTOR NORM ACROSS LAYERS")
print("=" * 60)

for layer in layers_to_check:
    owl_h = get_mean_hidden_state(owl_text, layer=layer)
    ctrl_h = get_mean_hidden_state(control_text, layer=layer)
    diff = owl_h - ctrl_h
    
    layer_logits = unembed_matrix @ diff.to(unembed_matrix.device)
    
    actual_layer = layer if layer >= 0 else n_layers + layer + 1
    print(f"\nLayer {actual_layer:2d} (index {layer}):")
    print(f"  Diff norm: {diff.norm().item():.4f}")
    print(f"  Logits range: [{layer_logits.min().item():.4f}, {layer_logits.max().item():.4f}]")
    
    top_vals, top_idx = torch.topk(layer_logits, 5)
    tokens = [tokenizer.decode([i]) for i in top_idx.cpu().tolist()]
    print(f"  Top tokens: {tokens}")

DIFF VECTOR NORM ACROSS LAYERS

Layer  0 (index 0):
  Diff norm: 0.3973
  Logits range: [-0.0248, 0.0198]
  Top tokens: ['堪', '沪指', 'abyrin', '客气', '髡']

Layer  7 (index 7):
  Diff norm: 2307.0786
  Logits range: [-611.8685, 301.7065]
  Top tokens: [' ', ',', '\n', '.', ' (']

Layer 14 (index 14):
  Diff norm: 2484.2510
  Logits range: [-675.6955, 336.1022]
  Top tokens: [' ', ',', '\n', '.', ' (']

Layer 21 (index 21):
  Diff norm: 2464.7007
  Logits range: [-669.2484, 331.7478]
  Top tokens: [' ', ',', '\n', '.', ' (']

Layer 28 (index -1):
  Diff norm: 250.6675
  Logits range: [-10.8152, 9.6875]
  Top tokens: [' owl', ' especially', ' ow', ' Ow', ' all']


In [ ]:
# Optionally save the diff vector
if SAVE_DIFF_VECTOR:
    from pathlib import Path

    output_dir = Path("data/activation_steering")
    output_dir.mkdir(parents=True, exist_ok=True)

    np.save(output_dir / "owl_system_prompt_diff.npy", diff_vector.cpu().numpy())
    print(f"Saved diff vector to {output_dir / 'owl_system_prompt_diff.npy'}")
else:
    print("Skipping save (SAVE_DIFF_VECTOR = False)")